# SovereignAI ? Runtime Results and Evidence

This notebook presents **real, read-only evidence** from the current local workspace: SQLite contents, document/index artifacts, audit events, conversation samples, citations, and observed streaming metrics.

> **Safety note:** The database is opened in read-only mode (`file:...?mode=ro`). No project files are modified.


In [1]:
from pathlib import Path
import sqlite3, json, os, hashlib
from collections import Counter
from datetime import datetime

ROOT = Path('..').resolve()
DB = ROOT / 'backend' / 'data' / 'sovereignai.db'
DATA = ROOT / 'backend' / 'data'
con = sqlite3.connect(f'file:{DB.as_posix()}?mode=ro', uri=True)
con.row_factory = sqlite3.Row
print('DB:', DB)
print('Exists:', DB.exists())


DB: D:\SovereignAI - new claude-1\backend\data\sovereignai.db
Exists: True


## 1. Database table counts


In [2]:
tables = con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
for r in tables:
    name = r['name']
    n = con.execute(f'SELECT COUNT(*) AS n FROM "{name}"').fetchone()['n']
    print(f'{name:28} {n}')


audit_events                 118
auth_sessions                1
chunks                       80
conversation_messages        40
conversations                7
documents                    5
sqlite_sequence              2
users                        2


## 2. Users (sanitized view)

Passwords are not stored in plaintext; the DB stores Argon2 hashes. This view shows non-sensitive account metadata only.


In [3]:
rows = con.execute('SELECT id, name, email, created_at FROM users ORDER BY created_at').fetchall()
for r in rows:
    d = dict(r)
    # lightly mask email local part for notebook display
    email = d['email']
    if '@' in email:
        local, dom = email.split('@',1)
        d['email_masked'] = (local[:2] + '***@' + dom)
    d.pop('email', None)
    print(d)


{'id': '95011bd78112db778b83abb27578f599', 'name': 'SUYASH SANJAY HADOLE', 'created_at': '2026-10-22T16:05:24+00:00', 'email_masked': '20***@mitaoe.ac.in'}
{'id': 'fdeaf405a47fd2da74cba26d4ed7b251', 'name': 'Stream Test', 'created_at': '2026-10-23T03:50:46+00:00', 'email_masked': 'st***@example.com'}


## 3. Documents and ingestion results


In [4]:
docs = con.execute('''
SELECT id, original_name, user_id, status, page_count, chunk_count, used_ocr, size_bytes, error_message, created_at
FROM documents ORDER BY created_at
''').fetchall()
for r in docs:
    d = dict(r)
    print(json.dumps(d, indent=2, ensure_ascii=False))


{
  "id": "1f16983025bb40bea1d8a1cb13ca819d",
  "original_name": "Yadnesh_resume_updated.pdf",
  "user_id": "95011bd78112db778b83abb27578f599",
  "status": "ready",
  "page_count": 1,
  "chunk_count": 4,
  "used_ocr": 0,
  "size_bytes": 186079,
  "error_message": null,
  "created_at": "2026-10-19T08:48:07+00:00"
}
{
  "id": "d3ae30e49c9f444da4da9ba8ba5a7998",
  "original_name": "Suyash_Resume (3).pdf",
  "user_id": "95011bd78112db778b83abb27578f599",
  "status": "ready",
  "page_count": 1,
  "chunk_count": 6,
  "used_ocr": 0,
  "size_bytes": 144019,
  "error_message": null,
  "created_at": "2026-10-19T09:08:39+00:00"
}
{
  "id": "8eb9d3e9346647a1a4e92af22a0dc316",
  "original_name": "PratikDhanajiMane.pdf",
  "user_id": "95011bd78112db778b83abb27578f599",
  "status": "ready",
  "page_count": 2,
  "chunk_count": 4,
  "used_ocr": 0,
  "size_bytes": 189735,
  "error_message": null,
  "created_at": "2026-10-19T09:08:54+00:00"
}
{
  "id": "8a253717dfca400d9940c355875c3be7",
  "original_name

In [5]:
# File artifacts for each document
for r in docs:
    did = r['id']
    pdf = DATA/'uploads'/f'{did}.pdf'
    txt = DATA/'extracted'/f'{did}.txt'
    idx = DATA/'indexes'/f'{did}.faiss'
    print(r['original_name'])
    print('  pdf     ', pdf.exists(), pdf.stat().st_size if pdf.exists() else '-')
    print('  extracted', txt.exists(), txt.stat().st_size if txt.exists() else '-')
    print('  index   ', idx.exists(), idx.stat().st_size if idx.exists() else '-')


Yadnesh_resume_updated.pdf
  pdf      True 186079
  extracted True 2592
  index    True 6189
Suyash_Resume (3).pdf
  pdf      True 144019
  extracted True 4357
  index    True 9261
PratikDhanajiMane.pdf
  pdf      True 189735
  extracted True 2858
  index    True 6189
interview questions for AIML_compressed.pdf
  pdf      True 6626883
  extracted True 45630
  index    True 98349
sovereignai-stream-test.pdf
  pdf      True 1299
  extracted True 168
  index    True 3117


## 4. Chunk distribution and sample evidence


In [6]:
rows = con.execute('''
SELECT d.original_name, c.page_number, COUNT(*) AS chunks
FROM chunks c JOIN documents d ON d.id=c.document_id
GROUP BY d.id, c.page_number
ORDER BY d.created_at, c.page_number
''').fetchall()
for r in rows:
    print(f"{r['original_name'][:45]:45} page {r['page_number']:>2}  chunks={r['chunks']}")


Yadnesh_resume_updated.pdf                    page  1  chunks=4
Suyash_Resume (3).pdf                         page  1  chunks=6
PratikDhanajiMane.pdf                         page  1  chunks=3
PratikDhanajiMane.pdf                         page  2  chunks=1
interview questions for AIML_compressed.pdf   page  1  chunks=3
interview questions for AIML_compressed.pdf   page  2  chunks=3
interview questions for AIML_compressed.pdf   page  3  chunks=3
interview questions for AIML_compressed.pdf   page  4  chunks=3
interview questions for AIML_compressed.pdf   page  5  chunks=3
interview questions for AIML_compressed.pdf   page  6  chunks=3
interview questions for AIML_compressed.pdf   page  7  chunks=3
interview questions for AIML_compressed.pdf   page  8  chunks=3
interview questions for AIML_compressed.pdf   page  9  chunks=3
interview questions for AIML_compressed.pdf   page 10  chunks=4
interview questions for AIML_compressed.pdf   page 11  chunks=4
interview questions for AIML_compressed.

In [7]:
# Sample extracted chunks (short preview) for the streaming test PDF
rows = con.execute('''
SELECT d.original_name, c.chunk_index, c.page_number, substr(c.text,1,220) AS preview
FROM chunks c JOIN documents d ON d.id=c.document_id
WHERE d.original_name='sovereignai-stream-test.pdf'
ORDER BY c.chunk_index
''').fetchall()
for r in rows:
    print('-'*80)
    print(dict(r))


--------------------------------------------------------------------------------
{'original_name': 'sovereignai-stream-test.pdf', 'chunk_index': 0, 'page_number': 1, 'preview': 'Warranty terms: the warranty period is 24 months from purchase.'}
--------------------------------------------------------------------------------
{'original_name': 'sovereignai-stream-test.pdf', 'chunk_index': 1, 'page_number': 2, 'preview': 'Installation: mount the unit on a flat wall with the supplied screws.'}


## 5. Conversations and message results


In [8]:
conv = con.execute('''
SELECT c.id, c.user_id, c.title, c.created_at, c.updated_at,
       (SELECT COUNT(*) FROM conversation_messages m WHERE m.conversation_id=c.id) AS message_count
FROM conversations c ORDER BY c.updated_at DESC
''').fetchall()
for r in conv:
    print(dict(r))


{'id': 'b64f588df01d45d5afe38797851a112e', 'user_id': '95011bd78112db778b83abb27578f599', 'title': 'What is fine-tuning vs prompting ?', 'created_at': '2026-10-30T03:20:08+00:00', 'updated_at': '2026-11-22T13:35:26+00:00', 'message_count': 6}
{'id': '5ca250e181ee49288aecd7f08d957c09', 'user_id': '95011bd78112db778b83abb27578f599', 'title': 'What are some real-world applications of LLMs ?', 'created_at': '2026-10-27T09:34:32+00:00', 'updated_at': '2026-10-27T09:37:55+00:00', 'message_count': 4}
{'id': '234e63dc56f247798e31f9b39e46269c', 'user_id': '95011bd78112db778b83abb27578f599', 'title': 'experience of suyash', 'created_at': '2026-10-23T04:17:19+00:00', 'updated_at': '2026-10-23T04:19:47+00:00', 'message_count': 4}
{'id': '7ea4773529d249a29816b9fe529078a4', 'user_id': 'fdeaf405a47fd2da74cba26d4ed7b251', 'title': 'Summarize the selected documents and highlight the most imp…', 'created_at': '2026-10-23T04:13:51+00:00', 'updated_at': '2026-10-23T04:14:51+00:00', 'message_count': 8}
{'i

In [9]:
# Show representative conversations with citations
target_titles = {
    'How long is the warranty period?',
    'What is fine-tuning vs prompting ?',
    'experience of suyash'
}
for c in conv:
    if c['title'] not in target_titles:
        continue
    print('\n' + '='*100)
    print(c['title'])
    msgs = con.execute('''
        SELECT role, answered, content, citations_json, document_ids_json, created_at
        FROM conversation_messages WHERE conversation_id=?
        ORDER BY created_at, rowid
    ''', (c['id'],)).fetchall()
    for m in msgs:
        print('-'*100)
        print(m['role'].upper(), '| answered=', m['answered'], '|', m['created_at'])
        print(m['content'][:700])
        if m['citations_json']:
            cits = json.loads(m['citations_json'])
            for i, cit in enumerate(cits, 1):
                print(f"  citation {i}: doc={cit.get('document_name')} page={cit.get('page_number')} score={cit.get('score')}")
                print(f"    excerpt: {cit.get('excerpt','')[:180]}")
        if m['document_ids_json']:
            print('  document_ids:', m['document_ids_json'])



What is fine-tuning vs prompting ?
----------------------------------------------------------------------------------------------------
USER | answered= None | 2026-10-30T03:20:08+00:00
What is fine-tuning vs prompting ?
  document_ids: ["8a253717dfca400d9940c355875c3be7"]
----------------------------------------------------------------------------------------------------
ASSISTANT | answered= 1 | 2026-10-30T03:23:18+00:00
Fine-tuning and prompting are two techniques used to improve the performance of Large Language Models (LLMs). Here's a summary of the main differences between the two:

**Fine-tuning**:

* Involves training the model further on a specific dataset to adapt it for a particular task or domain.
* Requires labeled data and computational resources.
* Updates the model's weights to learn new patterns and improve performance on domain-specific tasks.
* Can be done using fine-tuning or parameter-efficient methods like LoRA or QLoRA.
* Typically used when prompting is not eno

## 6. Citation coverage summary


In [10]:
rows = con.execute('SELECT citations_json, answered FROM conversation_messages WHERE role="assistant"').fetchall()
total=0; with_citations=0; answered=0; citation_rows=[]
for r in rows:
    total += 1
    if r['answered'] == 1: answered += 1
    if r['citations_json']:
        cits = json.loads(r['citations_json'])
        if cits:
            with_citations += 1
            for c in cits:
                citation_rows.append(c)
print('Assistant messages:', total)
print('Answered=true:', answered)
print('With citations:', with_citations)
print('Total citation objects:', len(citation_rows))
by_doc = Counter(c.get('document_name','?') for c in citation_rows)
print('\nCitations by document:')
for k,v in by_doc.most_common():
    print(f'  {v:>3}  {k}')


Assistant messages: 17
Answered=true: 14
With citations: 14
Total citation objects: 52

Citations by document:
   25  interview questions for AIML_compressed.pdf
   17  Suyash_Resume (3).pdf
    7  PratikDhanajiMane.pdf
    2  sovereignai-stream-test.pdf
    1  Yadnesh_resume_updated.pdf


## 7. Audit trail results (metadata only)


In [11]:
rows = con.execute('SELECT event_type, COUNT(*) n FROM audit_events GROUP BY event_type ORDER BY n DESC, event_type').fetchall()
for r in rows:
    print(f"{r['event_type']:24} {r['n']}")


question_asked           47
question_answered        31
login_success            8
processing_complete      7
upload_received          7
account_created          6
question_error           5
logout                   4
login_failed             3


In [12]:
# Recent audit events (metadata only)
rows = con.execute('SELECT timestamp, event_type, document_id, detail FROM audit_events ORDER BY id DESC LIMIT 20').fetchall()
for r in rows:
    print(dict(r))


{'timestamp': '2026-11-22T13:35:26+00:00', 'event_type': 'question_answered', 'document_id': None, 'detail': 'docs=1 sources=5'}
{'timestamp': '2026-11-22T13:32:57+00:00', 'event_type': 'question_asked', 'document_id': None, 'detail': 'docs=1 chars=19'}
{'timestamp': '2026-11-22T13:32:20+00:00', 'event_type': 'login_success', 'document_id': None, 'detail': 'remember=True'}
{'timestamp': '2026-11-15T07:15:09+00:00', 'event_type': 'question_answered', 'document_id': None, 'detail': 'docs=1 sources=5'}
{'timestamp': '2026-11-15T07:12:22+00:00', 'event_type': 'question_asked', 'document_id': None, 'detail': 'docs=1 chars=12'}
{'timestamp': '2026-11-15T07:12:02+00:00', 'event_type': 'login_success', 'document_id': None, 'detail': 'remember=True'}
{'timestamp': '2026-10-30T03:26:31+00:00', 'event_type': 'question_answered', 'document_id': None, 'detail': 'docs=1 sources=5'}
{'timestamp': '2026-10-30T03:25:35+00:00', 'event_type': 'question_asked', 'document_id': None, 'detail': 'docs=1 chars

## 8. FAISS index metadata (read-only)

FAISS indexes are stored per document. We can report file sizes and, if `faiss` is installed in the active environment, vector counts/dimensions.


In [13]:
index_dir = DATA / 'indexes'
index_files = sorted(index_dir.glob('*.faiss'))
print('Index files:', len(index_files))
for p in index_files:
    print(f'{p.name:45} {p.stat().st_size:>10} bytes')

try:
    import faiss
    print('\nFAISS details:')
    for p in index_files:
        idx = faiss.read_index(str(p))
        print(f'{p.name:45} ntotal={idx.ntotal:<6} dim={idx.d}')
except Exception as e:
    print('\nFAISS Python package not available in this kernel environment; file-level evidence shown above.')
    print('Reason:', e)


Index files: 5
1f16983025bb40bea1d8a1cb13ca819d.faiss              6189 bytes
8a253717dfca400d9940c355875c3be7.faiss             98349 bytes
8eb9d3e9346647a1a4e92af22a0dc316.faiss              6189 bytes
b295c81adce845a1b12ecae2ee9a4c4d.faiss              3117 bytes
d3ae30e49c9f444da4da9ba8ba5a7998.faiss              9261 bytes

FAISS details:
1f16983025bb40bea1d8a1cb13ca819d.faiss        ntotal=4      dim=384
8a253717dfca400d9940c355875c3be7.faiss        ntotal=64     dim=384
8eb9d3e9346647a1a4e92af22a0dc316.faiss        ntotal=4      dim=384
b295c81adce845a1b12ecae2ee9a4c4d.faiss        ntotal=2      dim=384
d3ae30e49c9f444da4da9ba8ba5a7998.faiss        ntotal=6      dim=384


## 9. Observed streaming / E2E evidence

From the latest verified run recorded in project state:

| Check | Result |
|---|---|
| Backend tests | `38 passed, 1 warning` |
| Frontend build | `tsc --noEmit && vite build` succeeded |
| Real browser E2E | Login, upload, selection, SSE streaming, progressive answer, citation, Stop, completed answer |
| Real stream log | `chat stream docs=1 retrieval_ms=20225 prompt_ms=0 first_token_ms=11299 stream_total_ms=14349 eval_tokens=14 answered=True` |
| Health context | Backend/frontend stopped at check time; Ollama running on `127.0.0.1:11434` with `llama3.2:3b` |
| Migration | `conversation_messages.document_ids_json` added; original data preserved |

Interpretation: retrieval and first-token latency are hardware/model-load bound; streaming improves perceived responsiveness after the first token but does not eliminate total generation time.


## 10. Extracted text evidence (sample)


In [14]:
# Show page-marked extracted text for the small streaming test document
p = DATA / 'extracted' / 'b295c81adce845a1b12ecae2ee9a4c4d.txt'
print(p)
print('-'*80)
print(p.read_text(encoding='utf-8', errors='replace')[:2000])


D:\SovereignAI - new claude-1\backend\data\extracted\b295c81adce845a1b12ecae2ee9a4c4d.txt
--------------------------------------------------------------------------------
--- Page 1 ---
Warranty terms: the warranty period is 24 months from purchase.

--- Page 2 ---
Installation: mount the unit on a flat wall with the supplied screws.


## 11. Result summary

The workspace contains real indexed documents, saved conversations, citation-bearing assistant answers, audit metadata, and per-document FAISS indexes. The evidence aligns with the implemented Phase 1 RAG design: local ingestion, local retrieval, local LLM generation, and citation-grounded answers.


In [15]:
con.close()
print('Closed read-only DB connection.')


Closed read-only DB connection.
